<h1>Prompting using LangChain</h1>

In [5]:
from langchain_openai import AzureChatOpenAI, ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

True

<div class="alert alert-success">Load environment variables</div>

In [6]:
endpoint = os.getenv("AZURE_ENDPOINT")
model = os.getenv("AZURE_OPENAI_MODEL")

subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

openai_llm = AzureChatOpenAI( 
        api_version=api_version,
        azure_deployment=model,
   )

In [7]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]

openai_llm.invoke(messages)

AIMessage(content="J'aime la programmation.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt_tokens': 30, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZBACah0qiRqXQoLbjbWaseApLrIi', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {'hate': {'filtered': False, 's

<div class="alert alert-success">Using PromptTemplate</div>

In [8]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template=""""
    Extract name, age and city of a fictional person. Name: , Age: , City: \n {format_instruction}. 
    """,
    input_variables=[],
)

chain = prompt_template | openai_llm
response = chain.invoke({"format_instruction":"Fifteen years old Ram lives in Delhi"})
response
response.content

'Name: Ram\nAge: 15\nCity: Delhi'

<h2 class="alert alert-info"> StrOutputParser as Output Parser</h2>

<div class="alert alert-success">chain = prompt_template | openai_llm | StrOutputParser()</div>

<div class="alert alert-block alert-success">Input: Fifteen years old Ram lives in Delhi</br>Output: 'Name: Ram, Age: 15, City: Delhi'</div>

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = PromptTemplate(
    template=""""
    Extract name, age and city of a fictional person. Name: , Age: , City: \n {format_instruction}. 
    """,
    input_variables=[],
)

chain = prompt_template | openai_llm | StrOutputParser()
response = chain.invoke({"format_instruction":"Fifteen years old Ram lives in Delhi"})
response

'Name: Ram\nAge: 15\nCity: Delhi'

<h2 class="alert alert-info"> JsonOutputParser as Output Parser</h2>

<div class="alert alert-block alert-info">
json_output_parser = JsonOutputParser() </br>
....</br>
chain = prompt_template | openai_llm | json_output_parser</div>

<div class="alert alert-block alert-success">Input: Fifteen years old Ram lives in Delhi. </br>
Output: {'name': 'Ram', 'age': 15, 'city': 'Delhi'}
</div>

<div class="alert alert-block alert-danger">
1. No Data type validation. Age could come as 15 or Fifteen.
</div>

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

json_output_parser = JsonOutputParser() 
prompt_template = PromptTemplate(
    template=""""
    Extract name, age and city of a fictional person from the {input}.\n {format_instruction}. 
    """,
    input_variables=["input"],
    partial_variables={'format_instruction': json_output_parser.get_format_instructions()}
)

chain = prompt_template | openai_llm | json_output_parser
response = chain.invoke({"input":"Fifteen years old Ram lives in Delhi"})
response

{'name': 'Ram', 'age': 15, 'city': 'Delhi'}

<div class="alert alert-block alert-info">
pydantic_output_parser = PydanticOutputParser() </br>
....</br>
chain = prompt_template | openai_llm | pydantic_output_parser</div>

<div class="alert alert-block alert-success">Input: Fifteen years old Ram lives in Delhi. </br>
Output: {'name': 'Ram', 'age': 15, 'city': 'Delhi'}
</div>

<div class="alert alert-block alert-success">
1. Data validation
</div>

In [17]:
from pydantic import BaseModel, Field

class Student(BaseModel):
    name: str = Field(description="Name of the person")
    age: int = Field(description="Age of the person", examples=[10])
    city: str = Field(description="Name of the City the person lives in", examples=['Bengaluru'])

In [23]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

pydantic_output_parser = PydanticOutputParser(pydantic_object=Student)

prompt_template = PromptTemplate(
    template=""""
    Extract name, age and city of a fictional person from the {input}. Make sure your answers are grounded.\n {format_instruction}. 
    """,
    input_variables=["input"],
    partial_variables={'format_instruction': pydantic_output_parser.get_format_instructions()}
)

chain = prompt_template | openai_llm | pydantic_output_parser
response = chain.invoke({"input":"Fifteen years old  lives in Delhi"})
response

Student(name='Unknown', age=15, city='Delhi')

In [24]:
prompt_template_not_grounded = PromptTemplate(
    template=""""
    Extract name, age and city of a fictional person from the {input}.\n {format_instruction}. 
    """,
    input_variables=["input"],
    partial_variables={'format_instruction': pydantic_output_parser.get_format_instructions()}
)

chain = prompt_template_not_grounded | openai_llm | pydantic_output_parser
response_not_grounded = chain.invoke({"input":"Fifteen years old  lives in Delhi"})
response_not_grounded

Student(name='Mira Singh', age=15, city='Delhi')